# Word to Vector (Word Embeddings)

After tokenization, each token must be converted into a numerical representation the model can compute with. **Word embeddings** are dense vectors in a continuous vector space that capture semantic meaning.

---

## 1. The Core Idea

Raw text → tokens → integers → **vectors**

```
"king"   →  token_id: 2300  →  [0.21, -0.45, 0.83, 0.12, ...]   (d-dimensional vector)
"queen"  →  token_id: 4815  →  [0.19, -0.41, 0.79, 0.34, ...]
"apple"  →  token_id: 7392  →  [-0.62, 0.91, -0.13, 0.05, ...]
```

Similar meanings → similar vectors → close together in vector space.

> **Distributional Hypothesis:** Words that appear in similar contexts tend to have similar meanings.
> — J.R. Firth, 1957: *"You shall know a word by the company it keeps."*

---

## 2. One-Hot Encoding (The Naive Baseline)

Before dense embeddings, words were represented as sparse one-hot vectors.

```
Vocabulary: [cat, dog, king, queen, apple]  → size = 5

"cat"   →  [1, 0, 0, 0, 0]
"dog"   →  [0, 1, 0, 0, 0]
"king"  →  [0, 0, 1, 0, 0]
```

### Problems

| Issue | Description |
|---|---|
| **Sparsity** | Vector size = vocabulary size (30k–100k dimensions) |
| **No similarity** | `dot("cat", "dog") = 0` — all words are equidistant |
| **No meaning** | No information about word relationships |
| **Curse of dimensionality** | Intractable for large vocabularies |

Dense embeddings solve all of these.

---

## 3. Word2Vec

Word2Vec (Mikolov et al., Google 2013) is the foundational algorithm for learning dense word embeddings. It trains a shallow neural network to predict words from context, and uses the learned weights as embeddings.

### Key Insight

The model is never the goal — the **weight matrix is**. The prediction task is a proxy to force the model to learn meaningful representations.

### Two Architectures

#### CBOW — Continuous Bag of Words

Predicts the **center word** from surrounding context words.

```
Context:  ["The", "cat", ___, "on", "the"]
Target:   "sat"

Input:  average of context word vectors
Output: predict center word
```

- Faster to train
- Better for frequent words
- Smooths over distributional information

#### Skip-gram

Predicts the **context words** from the center word.

```
Center:   "sat"
Targets:  ["The", "cat", "on", "the"]

Input:  center word vector
Output: predict each context word
```

- Slower to train
- Better for rare words and larger datasets
- More commonly used in practice

```
           CBOW                    Skip-gram
    [w(t-2), w(t-1),         w(t)  →  w(t-2)
     w(t+1), w(t+2)]                   w(t-1)
          ↓                            w(t+1)
         w(t)                          w(t+2)
```

### Training Objective

Maximize the probability of observing the actual context words given the center word (Skip-gram):

```
Maximize:  Σ log P(w_context | w_center)
```

This is computed using a softmax over the entire vocabulary — expensive for large vocabularies, solved using **negative sampling**.

---

## 4. Training Tricks

### Negative Sampling

Instead of updating weights for all vocabulary words on each step, only update:
- The 1 correct (positive) word
- K randomly sampled incorrect (negative) words (typically K = 5–20)

This reduces computation from O(V) to O(K) per step.

```
Positive sample:  ("sat", "cat")     → push vectors closer
Negative samples: ("sat", "pizza")   → push vectors apart
                  ("sat", "Monday")
                  ("sat", "river")
```

### Subsampling of Frequent Words

Very common words ("the", "a", "is") appear so often they dominate training but carry little semantic value. Each word `w` is discarded during training with probability:

```
P(discard) = 1 - sqrt(t / freq(w))    where t ≈ 1e-5
```

This speeds up training and improves embeddings for rare words.

### Window Size

The context window size `c` controls what counts as "context":

```
Sentence:  "The quick brown fox jumps over the lazy dog"
Center:    "fox"
Window=2:  context = ["quick", "brown", "jumps", "over"]
Window=1:  context = ["brown", "jumps"]
```

- Small window → syntactic relationships (POS, grammar)
- Large window → semantic/topical relationships

---

## 5. The Embedding Space

After training, each word has a vector in a continuous high-dimensional space (typically 100–300 dimensions). Geometry in this space encodes meaning.

### Semantic Similarity

```
cosine_similarity("king", "queen")  ≈  0.87   ← similar
cosine_similarity("king", "apple")  ≈  0.12   ← unrelated
```

Cosine similarity is preferred over Euclidean distance because it normalizes for vector magnitude.

### Vector Arithmetic (Analogy Reasoning)

The most famous property of Word2Vec:

```
king  -  man  +  woman  ≈  queen
paris - france + germany ≈ berlin
walking - walk + swim    ≈ swimming
```

This works because relational structure is encoded as **consistent directional offsets** in the space.

```
Vector space:

        queen ●              ● king
              ↑              ↑
         +woman            +man
              |              |
        woman ●              ● man

    king - man + woman ≈ queen
```

### Clusters

Related words cluster together:

```
[dog, cat, rabbit, hamster]   ← pets cluster
[Paris, London, Berlin, Tokyo] ← capital cities cluster
[run, walk, swim, jump]       ← motion verbs cluster
```

---

## 6. GloVe — Global Vectors for Word Representation

GloVe (Pennington et al., Stanford 2014) takes a different approach to the same goal.

### Key Difference

While Word2Vec uses a **local context window** (looks at one sentence at a time), GloVe uses **global co-occurrence statistics** across the entire corpus.

### How It Works

1. Build a co-occurrence matrix X where `X[i][j]` = how often word j appears near word i across the whole corpus
2. Train vectors to satisfy:

```
w_i · w_j + b_i + b_j  ≈  log( X[i][j] )
```

3. The dot product of two word vectors should equal the log of their co-occurrence count

### Word2Vec vs GloVe

| Aspect | Word2Vec | GloVe |
|---|---|---|
| Approach | Predictive (neural) | Count-based (matrix factorization) |
| Data usage | Local context windows | Global co-occurrence matrix |
| Training | Stochastic (SGD) | Batch (least squares) |
| Speed | Fast (negative sampling) | Can be slow for large corpora |
| Rare words | Good (skip-gram) | Weaker |
| Common words | Subsampling helps | Built into weighting function |
| Analogy tasks | Strong | Slightly stronger |

In practice, both produce high-quality embeddings. GloVe tends to win on analogy benchmarks; Word2Vec can be better on similarity tasks.

---

## 7. FastText

FastText (Bojanowski et al., Facebook 2017) extends Word2Vec to handle morphology.

### Key Idea

Instead of one vector per word, each word is represented as the **sum of its character n-gram vectors**.

```
"playing"  →  n-grams (n=3): ["<pl", "pla", "lay", "ayi", "yin", "ing", "ng>"]
              + whole word: "<playing>"

vector("playing") = sum of all n-gram vectors
```

### Advantages Over Word2Vec

| Feature | Word2Vec | FastText |
|---|---|---|
| OOV words | No vector | Approximates from n-grams |
| Rare words | Weak representation | Better (shares n-grams with similar words) |
| Morphology | Ignored | Captured ("play", "plays", "playing" share components) |
| Languages | English-centric | Works well for morphologically rich languages (Turkish, Finnish, Arabic) |

```
"playful"   (unseen word)
→ ["<pl", "pla", "lay", "ayu", "yfu", "ful", "ul>"]
→ vector approximated from seen words sharing these n-grams
```

---

## 8. Static vs. Contextual Embeddings

The critical limitation of Word2Vec, GloVe, and FastText: **each word has exactly one vector regardless of context**.

### The Polysemy Problem

```
"bank" → same vector for both:
  "I deposited money at the bank."    ← financial institution
  "We sat on the river bank."         ← edge of a river
```

### Static Embeddings

```
Word2Vec / GloVe / FastText:

"bank"  →  [0.34, -0.12, 0.88, ...]   ← one fixed vector, always
```

### Contextual Embeddings (ELMo, BERT, GPT)

```
BERT / GPT:

"bank" in "river bank"    →  [0.21, 0.95, -0.44, ...]
"bank" in "savings bank"  →  [0.87, -0.31, 0.62, ...]
                              ↑ different vectors, same word
```

The vector changes based on surrounding context — this is the key advance of transformer-based models.

| Model | Type | Context-aware | Year |
|---|---|---|---|
| Word2Vec | Static | ✗ | 2013 |
| GloVe | Static | ✗ | 2014 |
| FastText | Static | ✗ | 2017 |
| ELMo | Contextual (LSTM) | ✓ | 2018 |
| BERT | Contextual (Transformer) | ✓ | 2018 |
| GPT series | Contextual (Transformer) | ✓ | 2018– |

---

## 9. Embedding Layers in Modern LLMs

In transformer-based LLMs, the embedding is the first layer of the model.

### Architecture

```
Input text
    ↓
Tokenizer  →  [token_id_1, token_id_2, ..., token_id_n]
    ↓
Embedding Matrix  E  (shape: vocab_size × d_model)
    ↓
Token Vectors  [e_1, e_2, ..., e_n]   (each of dimension d_model)
    ↓
+ Positional Encoding
    ↓
Transformer Layers
```

### Embedding Matrix

The embedding matrix is a lookup table of shape `[vocab_size × d_model]`:

```
GPT-2:    [50,257 × 768]
GPT-3:    [50,257 × 12,288]
LLaMA-7B: [32,000 × 4,096]
BERT:     [30,522 × 768]
```

Each row is the embedding vector for one token. Looking up a token's embedding is just indexing into this matrix.

### Weight Tying

In many models, the **input embedding matrix and the output (unembedding) matrix are shared** (transposed). This halves parameter count and empirically improves performance.

```
Input:   token_id → embedding vector         (E)
Output:  hidden state → logits over vocab    (E^T)
```

---

## 10. Positional Encoding

Embeddings alone carry no information about **where** a token appears in the sequence. Positional encoding adds this.

### Sinusoidal (Original Transformer, 2017)

Fixed (not learned), uses sine and cosine at different frequencies:

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

### Learned Positional Embeddings

BERT and GPT-2 learn a separate positional embedding vector for each position, added to the token embedding:

```
final_embedding = token_embedding + positional_embedding
```

### Rotary Positional Encoding (RoPE)

Used by LLaMA, Mistral, GPT-NeoX. Instead of adding position info, it **rotates** query and key vectors in attention by an angle proportional to position. Generalizes better to sequences longer than those seen during training.

### ALiBi (Attention with Linear Biases)

Adds a position-dependent bias directly to attention scores rather than to embeddings. Used by MPT, BLOOM. Excellent length generalization.

| Method | Approach | Length generalization | Used By |
|---|---|---|---|
| Sinusoidal | Add fixed sin/cos | Moderate | Original Transformer |
| Learned | Add learned vectors | Poor (fixed max length) | BERT, GPT-2 |
| RoPE | Rotate Q/K in attention | Good | LLaMA, Mistral, GPT-NeoX |
| ALiBi | Bias attention scores | Very good | MPT, BLOOM |

---

## 11. Embedding Dimensions Across Models

Embedding dimension `d_model` determines the richness of representation:

| Model | d_model | Vocab Size | Parameters (embed only) |
|---|---|---|---|
| Word2Vec (typical) | 300 | 3M | ~900M |
| BERT-base | 768 | 30,522 | ~23M |
| BERT-large | 1,024 | 30,522 | ~31M |
| GPT-2 small | 768 | 50,257 | ~39M |
| GPT-3 | 12,288 | 50,257 | ~617M |
| LLaMA-7B | 4,096 | 32,000 | ~131M |
| LLaMA-70B | 8,192 | 32,000 | ~262M |

---

## 12. Key Properties of Good Embeddings

| Property | Description |
|---|---|
| **Semantic similarity** | Similar meanings → small cosine distance |
| **Analogy structure** | Relational offsets consistent across pairs |
| **Isotropy** | Vectors spread across the space, not collapsed into a cone |
| **Anisotropy (problem)** | BERT embeddings are known to be anisotropic — many dimensions are wasted |
| **Contextual sensitivity** | Same word → different vectors in different contexts (transformer models) |

---

## 13. Full Pipeline Summary

```
Raw Text
   ↓
Tokenizer (BPE / WordPiece / SentencePiece)
   ↓
Token IDs:  [2300, 4815, 7392, ...]
   ↓
Embedding Matrix Lookup   E[token_id]
   ↓
Token Vectors:  [[0.21, -0.45, ...], [0.19, -0.41, ...], ...]
   ↓
+ Positional Encoding (RoPE / Learned / ALiBi)
   ↓
Contextual Representations via Transformer Layers
   ↓
Output Logits  →  Softmax  →  Next Token Probabilities
```

---

## Quick Reference

| Term | Meaning |
|---|---|
| Embedding | Dense vector representation of a token |
| Word2Vec | Shallow neural net trained to predict context; weights = embeddings |
| CBOW | Predict center word from context |
| Skip-gram | Predict context words from center word |
| Negative sampling | Train on K random negatives instead of full softmax |
| GloVe | Embeddings from global co-occurrence statistics |
| FastText | Sub-word (n-gram) based embeddings; handles OOV |
| Static embedding | One vector per word, context-independent |
| Contextual embedding | Vector depends on surrounding context (BERT, GPT) |
| d_model | Embedding dimension size |
| Positional encoding | Adds position information to token embeddings |
| RoPE | Rotary positional encoding — rotates Q/K vectors |
| Weight tying | Sharing input embedding and output unembedding matrix |